# mnemoa: corpus study, precision check and benchmarks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Flotrpy/mnemoa/blob/main/notebooks/mnemoa_colab.ipynb)

This notebook:

1. builds mnemoa from source and installs the Python SDK,
2. scans a corpus of public repositories that have `AGENTS.md` or `CLAUDE.md` and computes per-rule prevalence,
3. samples findings into a table you label by hand to measure precision,
4. runs `mnemoa-bench` and plots the 12 suites,
5. runs `mnemoa forget` on three repositories and charts always-loaded tokens before and after,
6. saves CSVs and PNGs to `notebooks/results/`.

It runs on Colab and locally with Jupyter. Set `CORPUS_LIMIT` to a small number for a quick local run.

In [ ]:
import os, subprocess, sys, pathlib, json, shutil

IN_COLAB = "google.colab" in sys.modules
# Where the mnemoa checkout lives. Locally, run the notebook from inside the repo.
REPO = pathlib.Path(os.environ.get("MNEMOA_REPO", "/content/mnemoa" if IN_COLAB else pathlib.Path.cwd().parent)).resolve()
CORPUS_LIMIT = int(os.environ.get("CORPUS_LIMIT", "100"))
RESULTS = REPO / "notebooks" / "results"
WORK = pathlib.Path(os.environ.get("MNEMOA_WORK", "/content/corpus" if IN_COLAB else REPO / "target" / "corpus"))

def sh(cmd, check=True, **kw):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check, text=True, **kw)

print("repo:", REPO, "| colab:", IN_COLAB, "| corpus limit:", CORPUS_LIMIT)

## 1. Build

In [ ]:
if IN_COLAB and not (REPO / "Cargo.toml").exists():
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
    os.environ["PATH"] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ["PATH"]
    sh(f"git clone --depth 1 https://github.com/Flotrpy/mnemoa {REPO}")
sh(f"cd {REPO} && cargo build --release -p mnemoa-cli -p mnemoa-bench")
sh(f"{sys.executable} -m pip install -q -e {REPO}/sdk/python pandas matplotlib")
MNEMOA = str(REPO / "target" / "release" / "mnemoa")
BENCH = str(REPO / "target" / "release" / "mnemoa-bench")
sh(f"{MNEMOA} --version")

## 2. Corpus study

Edit `CORPUS` to change the sample. Repositories are shallow-cloned, so MNA005 (init fossilization, needs history) stays silent here.
Repositories that turn out to have no agent instruction files are recorded and skipped.

In [ ]:
CORPUS = [
    # Popular repositories likely to carry AGENTS.md or CLAUDE.md. Swap freely.
    "openai/codex", "openai/openai-agents-python", "openai/openai-agents-js", "anthropics/claude-code",
    "anthropics/anthropic-sdk-python", "anthropics/anthropic-sdk-typescript", "vercel/next.js", "vercel/ai",
    "microsoft/vscode", "microsoft/TypeScript", "microsoft/playwright", "microsoft/autogen",
    "langchain-ai/langchain", "langchain-ai/langgraph", "run-llama/llama_index", "huggingface/transformers",
    "huggingface/smolagents", "pydantic/pydantic", "pydantic/pydantic-ai", "fastapi/fastapi",
    "astral-sh/uv", "astral-sh/ruff", "denoland/deno", "oven-sh/bun", "biomejs/biome",
    "rust-lang/rust-analyzer", "tokio-rs/tokio", "zed-industries/zed", "tauri-apps/tauri", "sveltejs/svelte",
    "vuejs/core", "vitejs/vite", "vitest-dev/vitest", "facebook/react", "remix-run/react-router",
    "tailwindlabs/tailwindcss", "shadcn-ui/ui", "supabase/supabase", "prisma/prisma", "drizzle-team/drizzle-orm",
    "trpc/trpc", "TanStack/query", "withastro/astro", "nuxt/nuxt", "electron/electron", "n8n-io/n8n",
    "excalidraw/excalidraw", "calcom/cal.com", "mastra-ai/mastra", "modelcontextprotocol/servers",
    "modelcontextprotocol/typescript-sdk", "modelcontextprotocol/python-sdk", "google-gemini/gemini-cli",
    "cline/cline", "continuedev/continue", "github/spec-kit", "block/goose", "sst/opencode", "All-Hands-AI/OpenHands",
    "browser-use/browser-use", "openai/openai-python", "openai/openai-node", "ollama/ollama", "ggml-org/llama.cpp",
    "vllm-project/vllm", "pytorch/pytorch", "apache/airflow", "grafana/grafana", "hashicorp/terraform",
    "kubernetes/kubernetes", "golang/go", "nodejs/node", "python/cpython", "rust-lang/rust",
    "mui/material-ui", "storybookjs/storybook", "jestjs/jest", "eslint/eslint", "prettier/prettier",
    "withfig/autocomplete", "PostHog/posthog", "getsentry/sentry", "mattermost/mattermost", "appwrite/appwrite",
    "directus/directus", "strapi/strapi", "medusajs/medusa", "payloadcms/payload", "refinedev/refine",
    "nestjs/nest", "expressjs/express", "honojs/hono", "elysiajs/elysia", "dotnet/aspnetcore",
    "JetBrains/kotlin", "flutter/flutter", "home-assistant/core", "immich-app/immich", "jellyfin/jellyfin",
][:CORPUS_LIMIT]

AGENT_FILES = ["AGENTS.md", "CLAUDE.md", ".claude/CLAUDE.md", "CLAUDE.local.md", "GEMINI.md", ".cursorrules",
               ".github/copilot-instructions.md"]
AGENT_DIRS = [".claude/rules", ".cursor/rules", ".kiro/steering", ".clinerules", "memory-bank", "specs"]

WORK.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def clone(slug):
    dest = WORK / slug.replace("/", "__")
    if not dest.exists():
        r = sh(f"git clone -q --depth 1 https://github.com/{slug} {dest}", check=False, capture_output=True)
        if r.returncode != 0:
            return None
    return dest

def has_agent_files(path):
    return any((path / f).exists() for f in AGENT_FILES) or any((path / d).exists() for d in AGENT_DIRS)

rows, skipped = [], []
# Set MNEMOA_PRUNE_CLONES=1 when disk is short: each clone is deleted after scanning,
# except the first three with warnings or infos, which the forget demo below uses.
PRUNE_CLONES = os.environ.get("MNEMOA_PRUNE_CLONES") == "1"
demo_keep = []
for slug in CORPUS:
    path = clone(slug)
    if path is None:
        skipped.append((slug, "clone failed")); continue
    if not has_agent_files(path):
        skipped.append((slug, "no agent files"))
        if PRUNE_CLONES:
            shutil.rmtree(path, ignore_errors=True)
        continue
    scan = subprocess.run([MNEMOA, "scan", str(path), "--format", "json", "--no-history",
                           "--targets", "claude-code,codex,cursor,copilot,kiro,cline,gemini"],
                          capture_output=True, text=True)
    budget = subprocess.run([MNEMOA, "budget", str(path), "--format", "json",
                             "--targets", "claude-code,codex,cursor"], capture_output=True, text=True)
    if scan.returncode not in (0, 1):
        skipped.append((slug, "scan error: " + scan.stderr.strip()[:200]))
        if PRUNE_CLONES:
            shutil.rmtree(path, ignore_errors=True)
        continue
    rows.append({"repo": slug, "scan": json.loads(scan.stdout), "budget": json.loads(budget.stdout)})
    summary = rows[-1]["scan"]["summary"]
    if summary["warnings"] + summary["infos"] > 0 and len(demo_keep) < 3:
        demo_keep.append(slug)
    elif PRUNE_CLONES:
        shutil.rmtree(path, ignore_errors=True)

print(f"scanned {len(rows)} repositories; skipped {len(skipped)}")
for s in skipped[:20]:
    print("  skipped:", *s)

In [ ]:
import pandas as pd

RULES = [f"MNA{i:03d}" for i in range(1, 14)]
findings = pd.DataFrame([
    {"repo": r["repo"], **{k: f[k] for k in ("rule", "severity", "path", "line", "message", "snippet")}}
    for r in rows for f in r["scan"]["findings"]
])
findings.to_csv(RESULTS / "corpus_findings.csv", index=False)

per_repo = pd.DataFrame(
    [{"repo": r["repo"], **{rule: any(f["rule"] == rule for f in r["scan"]["findings"]) for rule in RULES}} for r in rows]
).set_index("repo")
prevalence = (per_repo.mean() * 100).round(1).rename("mnemoa % of repos")
prevalence.to_csv(RESULTS / "prevalence.csv")

budgets = pd.DataFrame([
    {"repo": r["repo"], "target": t["target"], "tokens": t["total_tokens"], "lines": t["total_lines"],
     "codex_dropped_bytes": t.get("dropped_bytes") or 0}
    for r in rows for t in r["budget"]["targets"]
])
budgets.to_csv(RESULTS / "budgets.csv", index=False)
print("repos with at least one finding:", round(per_repo.any(axis=1).mean() * 100, 1), "%")
prevalence

The SCAM 2026 paper (arXiv:2606.15828) reports prevalence on the 100 most-starred repositories with a root AGENTS.md or CLAUDE.md,
using LLM-based detectors. mnemoa uses deterministic, under-matching rules on a different sample, so the bars measure
**different things on different samples**: read them side by side, not as a comparison of accuracy.

In [ ]:
import matplotlib.pyplot as plt

SCAM = {"MNA002": 62, "MNA001": 42, "MNA003": 35, "MNA004": 28, "MNA005": 24, "MNA006": 16}
NAMES = {"MNA001": "context bloat", "MNA002": "lint leakage", "MNA003": "skill leakage", "MNA004": "conflicting",
         "MNA005": "init fossilization", "MNA006": "blind reference"}
ids = list(SCAM)
x = range(len(ids))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - 0.2 for i in x], [SCAM[k] for k in ids], width=0.4, label="SCAM 2026 (LLM detectors, top-100 repos)")
ax.bar([i + 0.2 for i in x], [prevalence.get(k, 0) for k in ids], width=0.4, label=f"mnemoa (deterministic, n={len(rows)}, shallow clones)")
ax.set_xticks(list(x), [NAMES[k] for k in ids], rotation=20)
ax.set_ylabel("% of repositories")
ax.set_title("Smell prevalence: different methods, different samples")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "prevalence.png", dpi=150)
plt.show()

## 3. Precision check

Up to 30 findings per rule go into `results/labels.csv`. Fill the `correct` column with `1` (true positive) or `0` (false positive),
save, and re-run the next cell. Nothing is reported until rows are labeled.

In [ ]:
labels_path = RESULTS / "labels.csv"
if not labels_path.exists() and len(findings):
    # pd.concat keeps the `rule` column; groupby().apply drops it on pandas 3.
    sample = pd.concat(g.sample(min(len(g), 30), random_state=0) for _, g in findings.groupby("rule"))
    sample.assign(correct="").to_csv(labels_path, index=False)
    print("wrote", labels_path, "with", len(sample), "rows to label")

labels = pd.read_csv(labels_path) if labels_path.exists() else pd.DataFrame(columns=["rule", "correct"])
labeled = labels[pd.to_numeric(labels["correct"], errors="coerce").notna()]
if len(labeled):
    precision = labeled.assign(correct=labeled["correct"].astype(int)).groupby("rule")["correct"].agg(["mean", "count"])
    precision["mean"] = (precision["mean"] * 100).round(1)
    precision.rename(columns={"mean": "precision %", "count": "labeled"}).to_csv(RESULTS / "precision.csv")
    display(precision)
else:
    print("no labels yet")

## 4. Benchmarks

In [ ]:
bench = json.loads(subprocess.run([BENCH, "--json"], capture_output=True, text=True, check=True).stdout)
print(bench["hardware"])
bdf = pd.DataFrame(bench["results"])
bdf["avg_ms"] = bdf["avg_ns"] / 1e6
bdf.to_csv(RESULTS / "bench.csv", index=False)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(bdf["name"], bdf["avg_ms"])
ax.set_xscale("log")
ax.set_xlabel("average latency (ms, log scale)")
ax.set_title(f"mnemoa-bench ({bench['hardware']['cpu']}, {bench['hardware']['profile']})")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(RESULTS / "bench.png", dpi=150)
plt.show()
bdf[["id", "name", "iterations", "avg_ms", "throughput", "unit"]]

## 5. Forget demo

In [ ]:
demo = demo_keep
records = []
for slug in demo:
    path = WORK / slug.replace("/", "__")
    plan = json.loads(subprocess.run([MNEMOA, "forget", str(path), "--format", "json"],
                                     capture_output=True, text=True, check=True).stdout)
    for t in plan["tokens"]:
        records.append({"repo": slug, "target": t["target"], "before": t["before"], "after": t["after"],
                        "actions": len(plan["actions"])})
forget = pd.DataFrame(records)
forget.to_csv(RESULTS / "forget.csv", index=False)
if len(forget):
    pivot = forget.pivot_table(index=["repo", "target"], values=["before", "after"])[["before", "after"]]
    ax = pivot.plot.barh(figsize=(9, 5))
    ax.set_xlabel("always-loaded tokens (o200k_base estimate)")
    ax.set_title("mnemoa forget: before and after (dry run)")
    plt.tight_layout()
    plt.savefig(RESULTS / "forget.png", dpi=150)
    plt.show()
forget

## 6. Results

Everything above is in `notebooks/results/`: `prevalence.csv/png`, `corpus_findings.csv`, `budgets.csv`, `labels.csv`,
`precision.csv` (after labeling), `bench.csv/png`, `forget.csv/png`. Commit them to update the README.